In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

df = pd.read_csv('raw/raw_data.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print("Shape", df.shape)
print("\nColumns")
print(df.columns.tolist())

print("\nInfo")
print(df.info())

print('\nDescribe')
display(df.describe())

Shape (7043, 21)

Columns
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Info
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='str')

In [4]:
df.duplicated().sum()
df['customerid'].duplicated().sum()


np.int64(0)

In [5]:
missing_values = df.isnull().sum()
missing_procent =(df.isna().mean()*100).round(2)

missing_df = pd.DataFrame({
    'missing_values': missing_values, 
    'missing_procent': missing_procent
    }).sort_values(by='missing_procent', ascending=False)

display(missing_df)

,missing_values,missing_procent
customerid,0,0.0
deviceprotection,0,0.0
totalcharges,0,0.0
monthlycharges,0,0.0
paymentmethod,0,0.0
paperlessbilling,0,0.0
contract,0,0.0
streamingmovies,0,0.0
streamingtv,0,0.0
techsupport,0,0.0


In [6]:
blank_count = {}
for col in df.columns:
    blank_count[col] = df[col].astype(str).str.strip().eq('').sum()
blank_count_df = pd.DataFrame.from_dict(blank_count, orient='index', columns=['blank_count'])
display(blank_count_df.sort_values(by='blank_count', ascending=False))

,blank_count
totalcharges,11
customerid,0
deviceprotection,0
monthlycharges,0
paymentmethod,0
paperlessbilling,0
contract,0
streamingmovies,0
streamingtv,0
techsupport,0


In [7]:
df['totalcharges']=pd.to_numeric(df['totalcharges'], errors='coerce')
df['totalcharges'].isna().sum()

np.int64(11)

In [8]:
df[df['totalcharges'].isna()].head()
df = df.dropna(subset=['totalcharges']).copy()
df['totalcharges'].isna().sum()

np.int64(0)

'TotalCharges' contained black strings, whice were converted to missing values using 'pd.to_numeric()'. Since these rows correspond to customers with zeor tenure, they were removed from the dataset.

In [9]:
df = df.drop(columns=['customerid'])

In [10]:
df.nunique()

gender                 2
seniorcitizen          2
partner                2
dependents             2
tenure                72
phoneservice           2
multiplelines          3
internetservice        3
onlinesecurity         3
onlinebackup           3
deviceprotection       3
techsupport            3
streamingtv            3
streamingmovies        3
contract               3
paperlessbilling       2
paymentmethod          4
monthlycharges      1584
totalcharges        6530
churn                  2
dtype: int64

In [11]:
df['gender'] = df['gender'].map({"Male": 1 , "Female": 0})
binary_columns = ['partner', 'dependents', 'phoneservice', 'paperlessbilling', 'churn']
for col in binary_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
    

In [12]:
internet_col = ['onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']
for col in internet_col:
    df[col] = df[col].map({'No': 0, 'Yes': 1, 'No internet service': 0})
df['multiplelines'] = df['multiplelines'].map({'No': 0, 'Yes': 1, 'No phone service': 0})
df.nunique()

gender                 2
seniorcitizen          2
partner                2
dependents             2
tenure                72
phoneservice           2
multiplelines          2
internetservice        3
onlinesecurity         2
onlinebackup           2
deviceprotection       2
techsupport            2
streamingtv            2
streamingmovies        2
contract               3
paperlessbilling       2
paymentmethod          4
monthlycharges      1584
totalcharges        6530
churn                  2
dtype: int64

In [13]:
df = df.drop(columns=['totalcharges'])
df.head()

,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,churn
0,0,0,1,0,1,0,0,DSL,0,1,0,0,0,0,Month-to-month,1,Electronic check,29.85,0
1,1,0,0,0,34,1,0,DSL,1,0,1,0,0,0,One year,0,Mailed check,56.95,0
2,1,0,0,0,2,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,Mailed check,53.85,1
3,1,0,0,0,45,0,0,DSL,1,0,1,1,0,0,One year,0,Bank transfer (automatic),42.30,0
4,0,0,0,0,2,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,1


In [14]:
df.to_csv('processed/processed_data.csv', index=False)